# NOTEBOOK : 01_season_features.ipynb
# INPUT    : ../../data/raw/master_player_history.csv (fallback: ../../data/processed/player_gw.csv)
# OUTPUT   : ../../data/processed/season_features.csv

In [1]:
from pathlib import Path
import unicodedata
import numpy as np
import pandas as pd

ROOT = Path("../..")
RAW_PATH = ROOT / "data" / "raw" / "master_player_history.csv"
FALLBACK_PATH = ROOT / "data" / "processed" / "player_gw.csv"
OUT_PATH = ROOT / "data" / "processed" / "season_features.csv"

src_path = RAW_PATH if RAW_PATH.exists() else FALLBACK_PATH
print(f"Loading player history from: {src_path}")
df = pd.read_csv(src_path, low_memory=False)


def normalize_text(val: str) -> str:
    if pd.isna(val):
        return ""
    text = unicodedata.normalize("NFKD", str(val)).encode("ascii", "ignore").decode("ascii")
    return " ".join(text.lower().strip().split())


def standardize_position(val) -> str:
    s = str(val).strip().upper()
    mapping = {
        "1": "GK", "GKP": "GK", "GK": "GK", "GOALKEEPER": "GK",
        "2": "DEF", "DEF": "DEF", "DEFENDER": "DEF",
        "3": "MID", "MID": "MID", "MIDFIELDER": "MID",
        "4": "FWD", "FWD": "FWD", "FORWARD": "FWD",
    }
    return mapping.get(s, "MID")

Loading player history from: ..\..\data\processed\player_gw.csv


In [2]:
# 1. Harmonize column names gracefully
if "season" not in df.columns and "season_name" in df.columns:
    df["season"] = df["season_name"]

if "player_key" not in df.columns:
    name_col = next((c for c in ["name", "player_name", "web_name"] if c in df.columns), None)
    df["name"] = df[name_col].astype(str)
    df["player_key"] = df["name"].apply(normalize_text)
elif "name" not in df.columns:
    df["name"] = df["player_key"].str.title()

pos_col = next((c for c in ["position", "element_type"] if c in df.columns), None)
df["position"] = df[pos_col].apply(standardize_position) if pos_col else "MID"

team_col = next((c for c in ["team", "team_name"] if c in df.columns), None)
df["team"] = df[team_col].astype(str) if team_col else "Unknown"

price_col = next((c for c in ["value", "now_cost", "price"] if c in df.columns), None)
if price_col:
    df["price"] = pd.to_numeric(df[price_col], errors="coerce").fillna(50.0)
    # Convert FPL tenths (e.g. 125 -> 12.5) if values are > 25
    if df["price"].median() > 25:
        df["price"] = df["price"] / 10.0
else:
    df["price"] = 5.0

for col in ["minutes", "total_points", "goals_scored", "assists", "bps", "ict_index"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)
    else:
        df[col] = 0.0

In [3]:
# 2. Aggregate to one row per (player_key, season)
season_df = (
    df.groupby(["player_key", "season"], as_index=False)
    .agg(
        name=("name", "last"),
        team=("team", "last"),
        position=("position", "last"),
        minutes=("minutes", "sum"),
        total_points=("total_points", "sum"),
        goals_scored=("goals_scored", "sum"),
        assists=("assists", "sum"),
        bps=("bps", "sum"),
        ict_index=("ict_index", "sum"),
        price=("price", "last"),
        games_played=("minutes", lambda s: int((s > 0).sum())),
    )
    .sort_values(["player_key", "season"])
    .reset_index(drop=True)
)

In [4]:
# 3. Build strictly prior-season features (shift by 1 season per player)
g = season_df.groupby("player_key", group_keys=False)
season_df["prev_season"] = g["season"].shift(1)
season_df["prev_minutes"] = g["minutes"].shift(1)
season_df["prev_total_points"] = g["total_points"].shift(1)
season_df["prev_goals"] = g["goals_scored"].shift(1)
season_df["prev_assists"] = g["assists"].shift(1)
season_df["prev_bps"] = g["bps"].shift(1)
season_df["prev_ict_index"] = g["ict_index"].shift(1)
season_df["prev_price"] = g["price"].shift(1)
season_df["prev_games_played"] = g["games_played"].shift(1)

# Derived per-90 efficiency from prior season (safe division)
prev_90s = (season_df["prev_minutes"] / 90.0).replace(0, np.nan)
season_df["prev_points_per_90"] = (season_df["prev_total_points"] / prev_90s).fillna(0.0)
season_df["prev_goal_involvements_per_90"] = (
    (season_df["prev_goals"] + season_df["prev_assists"]) / prev_90s
).fillna(0.0)

# One-hot encode position (GK, DEF, MID, FWD)
pos_dummies = pd.get_dummies(season_df["position"], prefix="pos", dtype=int)
for col in ["pos_GK", "pos_DEF", "pos_MID", "pos_FWD"]:
    if col not in pos_dummies.columns:
        pos_dummies[col] = 0
season_df = pd.concat([season_df, pos_dummies[["pos_GK", "pos_DEF", "pos_MID", "pos_FWD"]]], axis=1)

In [5]:
# 4. Keep only valid season-to-season transitions (player played in prior season)
transitions = season_df.dropna(subset=["prev_total_points"]).copy()
transitions = transitions[transitions["prev_minutes"] >= 90].copy()  # filter 1-minute cameos
transitions["target_season_points"] = transitions["total_points"].astype(float)

transitions.to_csv(OUT_PATH, index=False)
print(f"Saved {len(transitions):,} season-to-season rows -> {OUT_PATH}")
print("Target seasons:", sorted(transitions["season"].unique()))
print(transitions[["name", "prev_season", "season", "prev_total_points", "target_season_points"]].head(8))

Saved 2,379 season-to-season rows -> ..\..\data\processed\season_features.csv
Target seasons: ['2021-22', '2022-23', '2023-24', '2024-25', '2025-26', '2026-27']
               name prev_season   season  prev_total_points  \
4    Aaron Connolly     2020-21  2021-22               38.0   
5    Aaron Connolly     2021-22  2023-24                5.0   
7   Aaron Cresswell     2020-21  2021-22              153.0   
8   Aaron Cresswell     2021-22  2022-23              115.0   
9   Aaron Cresswell     2022-23  2023-24               74.0   
10  Aaron Cresswell     2023-24  2024-25               10.0   
12     Aaron Hickey     2022-23  2023-24               63.0   
13     Aaron Hickey     2023-24  2024-25               17.0   

    target_season_points  
4                    5.0  
5                    0.0  
7                  115.0  
8                   74.0  
9                   10.0  
10                  36.0  
12                  17.0  
13                   0.0  
